# 04 — LoRA Fine-tuning: Thai Quiz Generator

**Teacher-Student Distillation:**
- Teacher: `Qwen2.5-14B` generates high-quality dataset (notebook 01 + slurm_teacher.sh)
- Student: `Qwen2.5-7B` learns via LoRA → faster inference + better Thai

**Requirements:**
- Dataset in `../dataset/generated/` (run notebook 01 first)
- GPU: 3g.40gb MIG (40GB VRAM) — 7B 4-bit + LoRA r=64 ≈ 12GB
- `unsloth` for 2× faster training

> Acknowledgement: Computational resources supported by KU Nontri AI, Kasetsart University

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

def pkg_installed(name):
    try:
        __import__(name)
        return True
    except ImportError:
        return False

# unsloth — brings in transformers, peft, accelerate
if not pkg_installed('unsloth'):
    print('Installing unsloth...')
    pip_install('unsloth[cu124-torch240]')
else:
    print('✅ unsloth already installed')

# trl ≥0.12 — needed for SFTTrainer eval_dataset + EarlyStoppingCallback
import importlib.metadata as _meta
try:
    _trl_ver = tuple(int(x) for x in _meta.version('trl').split('.')[:2])
    if _trl_ver < (0, 12):
        raise ValueError('too old')
    print(f'✅ trl {_meta.version("trl")} already installed')
except Exception:
    print('Installing trl>=0.12...')
    pip_install('trl>=0.12.0')

# datasets
if not pkg_installed('datasets'):
    pip_install('datasets')
else:
    print('✅ datasets already installed')

import os, json
from pathlib import Path
print('✅ All dependencies ready')

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────
import os, json
from pathlib import Path

# ปิด dynamo ตั้งแต่ต้น — ป้องกัน crash ระหว่าง training + save
os.environ['TORCHDYNAMO_DISABLE']   = '1'
os.environ['TORCH_COMPILE_DISABLE'] = '1'

# อ่าน base model จากไฟล์ที่ download_typhoon.sh บันทึกไว้
_info_file = Path.home() / 'ku_prep_arena/ai/models/downloaded_base_model.json'
if _info_file.exists():
    _info = json.loads(_info_file.read_text())
    BASE_MODEL = _info['path']
    print(f'✅ Using downloaded model: {_info["model_id"]}')
    print(f'   Path: {BASE_MODEL}')
else:
    BASE_MODEL = 'openthaigpt/openthaigpt1.5-7b-instruct'
    print(f'⚠️  downloaded_base_model.json not found — using {BASE_MODEL}')

# ── Paths: เขียนทุกอย่างลง /tmp (local SSD บน dgx) ─────────────────────
# NFS home dir ไม่รองรับ mmap write ไฟล์ใหญ่ → EPROTO (os error 71)
TMP_CKPT_DIR  = Path('/tmp/ku_typhoon_v1_ckpt')   # checkpoints ระหว่าง train
TMP_LORA_DIR  = Path('/tmp/ku_typhoon_v1')         # LoRA adapter สุดท้าย
HOME_LORA_DIR = Path.home() / 'ku_prep_arena/ai/models/ku_typhoon_v1'  # copy กลับหลัง train

RATED_DIR     = Path('../dataset/rated')
GEN_DIR       = Path('../dataset/generated')
MAX_SEQ_LEN   = 4096
LORA_RANK     = 64
BATCH_SIZE    = 4
GRAD_ACCUM    = 4
MAX_STEPS     = 300
LEARNING_RATE = 2e-4

BINARY_GAME_TYPES = {'flappy'}

# ── Chat template: ChatML (Qwen / Typhoon2.5-Qwen3) ──────────────────────
def format_chat(instruction: str, response: str) -> str:
    return (
        f'<|im_start|>user\n{instruction}<|im_end|>\n'
        f'<|im_start|>assistant\n{response}<|im_end|>'
    )

# ── Auto-select dataset dir ──────────────────────────────────────────────
rated_files = list(RATED_DIR.glob('*.json')) if RATED_DIR.exists() else []
if rated_files:
    DATASET_DIR = RATED_DIR
    print(f'✅ Dataset: {DATASET_DIR} ({len(rated_files)} files)')
else:
    DATASET_DIR = GEN_DIR
    print(f'⚠️  No rated dataset — using: {DATASET_DIR}')

TMP_CKPT_DIR.mkdir(parents=True, exist_ok=True)
TMP_LORA_DIR.mkdir(parents=True, exist_ok=True)
HOME_LORA_DIR.mkdir(parents=True, exist_ok=True)
print(f'Checkpoints : {TMP_CKPT_DIR}  (local SSD)')
print(f'LoRA output : {TMP_LORA_DIR}  (local SSD)')
print(f'Home copy   : {HOME_LORA_DIR} (NFS, written after train)')

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────
import re, json
from pathlib import Path
from collections import defaultdict

GAME_HINTS = {
    "flappy":  "FLAPPY BIRD: Generate TRUE/FALSE or binary-choice questions (exactly 2 choices: A and B). Questions must be short (≤12 words). One choice is clearly correct, one is clearly wrong.",
    "racer":   "SPEED RACER: Questions must be short (≤12 words). Provide exactly 4 choices (A/B/C/D).",
    "shooter": "SPACE SHOOTER: Use 'Which term/concept describes...?' format. Provide exactly 4 choices (A/B/C/D).",
    "snake":   "SNAKE QUIZ: Sequential or process-based questions. Provide exactly 4 choices (A/B/C/D).",
    "bricks":  "BRICK BREAKER: Definition-style questions ('What is...?'). Provide exactly 4 choices (A/B/C/D).",
}

DIFF_LABELS = {1: 'easy', 2: 'medium', 3: 'hard'}

def get_difficulty(q):
    return int(q.get('difficulty_teacher') or q.get('difficulty') or 2)

def strip_choice_prefix(text):
    return re.sub(r'^[A-Da-d][).]\s*', '', text).strip()

def filter_flappy_to_2_choices(questions):
    result = []
    for q in questions:
        q = dict(q)
        raw = q.get('choices', [])
        correct_idx = int(q.get('correct', 0))
        plain = [strip_choice_prefix(c) for c in raw]
        if len(plain) < 2:
            continue
        if len(plain) == 2:
            q['choices'] = [f"A. {plain[0]}", f"B. {plain[1]}"]
        else:
            correct_text = plain[correct_idx]
            wrong = [c for i, c in enumerate(plain) if i != correct_idx]
            q['choices'] = [f"A. {wrong[0]}", f"B. {correct_text}"]
            q['correct'] = 1
        result.append(q)
    return result

def format_example(text, game_type, questions):
    hint = GAME_HINTS.get(game_type, 'Provide exactly 4 choices (A/B/C/D).')
    n_choices = 2 if game_type in BINARY_GAME_TYPES else 4
    instruction = (
        f'You are a Thai/English quiz question generator.\n'
        f'{hint}\n'
        f'Create exactly 10 multiple-choice questions from the study material below.\n'
        f'Each question must have exactly {n_choices} choices.\n'
        f'Return ONLY a JSON array.\n\n'
        f'Study Material:\n{text[:3000]}'
    )
    response = json.dumps(questions, ensure_ascii=False)
    return {
        'text': format_chat(instruction, response),
        'game_type': game_type,
    }

# ── Load files ────────────────────────────────────────────────────────────
examples = []
diff_counts = {1: 0, 2: 0, 3: 0}
json_files = sorted(DATASET_DIR.glob('*.json'))
print(f'Found {len(json_files)} dataset files in {DATASET_DIR}')

groups = defaultdict(list)
for jf in json_files:
    parts = jf.stem.rsplit('_', 1)
    game_type = parts[-1] if len(parts) == 2 else 'general'
    source_stem = parts[0] if len(parts) == 2 else jf.stem
    groups[(source_stem, game_type)].append(jf)

for (source_stem, game_type), files in sorted(groups.items()):
    for jf in files:
        try:
            questions = json.loads(jf.read_text(encoding='utf-8'))
            if not questions:
                continue
            if game_type in BINARY_GAME_TYPES:
                questions = filter_flappy_to_2_choices(questions)
            else:
                filtered = []
                for q in questions:
                    raw = q.get('choices', [])
                    if len(raw) >= 4:
                        plain = [strip_choice_prefix(c) for c in raw[:4]]
                        q = dict(q)
                        q['choices'] = [f"{l}. {t}" for l, t in zip('ABCD', plain)]
                        filtered.append(q)
                questions = filtered
            for q in questions:
                diff_counts[get_difficulty(q)] += 1
            proxy_text = ' '.join(
                q.get('question', '') + ' ' + ' '.join(q.get('choices', []))
                for q in questions[:5]
            )
            examples.append(format_example(proxy_text, game_type, questions))
        except Exception as e:
            print(f'  Skip {jf.name}: {e}')

print(f'\nLoaded {len(examples)} training examples')
total_q = sum(diff_counts.values())
if total_q:
    print('Difficulty distribution:')
    for level in [1, 2, 3]:
        n = diff_counts[level]
        print(f'  {DIFF_LABELS[level]:6s} ({level}): {n:4d}  {n/total_q*100:.1f}%')


In [ ]:
# ── Load model with unsloth ──────────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

print(f'Loading {BASE_MODEL} with unsloth (4-bit)...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # auto
)
print('✅ Model loaded')

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(f'✅ LoRA adapters added (r={LORA_RANK})')

In [ ]:
# ── Prepare dataset ──────────────────────────────────────────────────────
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import is_bfloat16_supported
import random

if len(examples) == 0:
    raise ValueError('No training examples found! Run notebook 01 first.')

# ── Train / eval split 90/10 ──────────────────────────────────────────────
random.seed(42)
shuffled = examples[:]
random.shuffle(shuffled)
split = max(1, int(len(shuffled) * 0.1))
eval_examples  = shuffled[:split]
train_examples = shuffled[split:]
print(f'Train: {len(train_examples)} | Eval: {len(eval_examples)} examples')

train_dataset = Dataset.from_list(train_examples)
eval_dataset  = Dataset.from_list(eval_examples)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,   # ปิด packing — บางรุ่น trl ไม่รองรับ packing + eval พร้อมกัน
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        save_strategy='steps',
        save_steps=100,
        eval_strategy='steps',
        eval_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        lr_scheduler_type='cosine',
        optim='adamw_8bit',
        weight_decay=0.01,
        seed=42,
        output_dir=str(TMP_CKPT_DIR),   # /tmp — ไม่ใช่ NFS
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print(f'✅ Trainer ready — checkpoints → {TMP_CKPT_DIR}')

In [ ]:
# ── Validate training data integrity (before train) ──────────────────────
print('Validating training examples...')
errors = []
flappy_ok = 0
other_ok = 0

for i, ex in enumerate(train_examples + eval_examples):
    game_type = ex.get('game_type', 'unknown')
    expected_choices = 2 if game_type in BINARY_GAME_TYPES else 4

    try:
        import json as _json, re as _re
        assistant_start = ex['text'].index('<|im_start|>assistant\n') + len('<|im_start|>assistant\n')
        assistant_end = ex['text'].rindex('<|im_end|>')
        questions = _json.loads(ex['text'][assistant_start:assistant_end].strip())
    except Exception as e:
        errors.append(f'Example {i} ({game_type}): parse error — {e}')
        continue

    for q in questions:
        choices = q.get('choices', [])
        correct = q.get('correct')
        if len(choices) != expected_choices:
            errors.append(f'Example {i} Q{q.get("id","?")} ({game_type}): expected {expected_choices} choices, got {len(choices)}')
        if correct is None or not (0 <= int(correct) < len(choices)):
            errors.append(f'Example {i} Q{q.get("id","?")} ({game_type}): correct={correct} out of bounds')

    if game_type in BINARY_GAME_TYPES:
        flappy_ok += 1
    else:
        other_ok += 1

if errors:
    print(f'⚠️  {len(errors)} validation errors:')
    for err in errors[:10]:
        print(f'  ✗ {err}')
    raise ValueError('Fix dataset errors before training.')
else:
    print(f'✅ Validation passed — flappy: {flappy_ok}, other: {other_ok}')

In [ ]:
# ── Train ────────────────────────────────────────────────────────────────
import time
print('Starting training...')
t0 = time.time()

trainer_stats = trainer.train()

elapsed = time.time() - t0
print(f'\n✅ Training done in {elapsed/60:.1f} min')
print(f'   Final loss: {trainer_stats.training_loss:.4f}')

In [ ]:
# ── Save LoRA adapter → /tmp (local SSD) ─────────────────────────────────
# NFS home ไม่รองรับ write ไฟล์ใหญ่ (EPROTO) — save ใน /tmp แล้ว merge ใน slurm_finetune.sh
from pathlib import Path

print(f'Saving LoRA adapter to {TMP_LORA_DIR} (local SSD)...')
trainer.save_model(str(TMP_LORA_DIR))
tokenizer.save_pretrained(str(TMP_LORA_DIR))

saved = list(TMP_LORA_DIR.glob('*.json')) + list(TMP_LORA_DIR.glob('*.safetensors'))
if not saved:
    raise RuntimeError(f'Save failed — no files in {TMP_LORA_DIR}')
print(f'✅ LoRA saved to {TMP_LORA_DIR}:')
for f in saved:
    print(f'   {f.name}  ({f.stat().st_size/1024/1024:.1f} MB)')
print()
print('slurm_finetune.sh จะ merge LoRA ต่อบน node นี้ → /tmp/ku_typhoon_v1_merged')

In [ ]:
# ── Quick inference test ─────────────────────────────────────────────────
FastLanguageModel.for_inference(model)

test_prompt = (
    '<|im_start|>user\n'
    'คุณเป็นผู้สร้างคำถามแบบเลือกตอบสำหรับนิสิตมหาวิทยาลัยเกษตรศาสตร์\n'
    'FLAPPY BIRD: สร้างคำถาม 3 ข้อ แบบ True/False (2 ตัวเลือก A และ B) จากเนื้อหาต่อไปนี้:\n\n'
    'การเรียนรู้ของเครื่อง (Machine Learning) คือกระบวนการที่คอมพิวเตอร์เรียนรู้จากข้อมูล '
    'โดยไม่ต้องตั้งโปรแกรมตายตัว ใช้อัลกอริทึมเช่น Decision Tree, SVM, Neural Network\n\n'
    'ตอบเป็น JSON array เท่านั้น<|im_end|>\n'
    '<|im_start|>assistant\n'
)

inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
with __import__('torch').no_grad():
    output = model.generate(**inputs, max_new_tokens=400, temperature=0.5, do_sample=True)

decoded = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Model output:')
print(decoded[:600])
